# Pipeline utilities

Shared code for the pipeline: table definitions, execution logging, quarantine,
MERGE (upsert) logic and the parameterized `run_pipeline` function.
Loaded with `%run ./pipeline_utils` from `run_pipeline` and `pipeline_tests`.

In [0]:
%run ./layers

In [0]:
import os
import glob
import uuid
from datetime import datetime, timezone

from pyspark.sql.types import LongType

DEFAULT_SCHEMA = "world_bank"

# Default raw files for each load type (relative to the repo folder)
DEFAULT_SOURCES = {
    "full": "full_load_part*.json",
    "incremental": "incremental_full.json",
}


def utc_now():
    return datetime.now(timezone.utc)


def get_tables(catalog="", schema=DEFAULT_SCHEMA):
    catalog = (catalog or "").strip() or spark.catalog.currentCatalog()
    schema = (schema or "").strip() or DEFAULT_SCHEMA
    prefix = f"{catalog}.{schema}"
    return {
        "schema": prefix,
        "bronze": f"{prefix}.bronze_indicator_values",
        "silver": f"{prefix}.silver_indicator_values",
        "revisions": f"{prefix}.silver_indicator_revisions",
        "quarantine": f"{prefix}.quarantine_records",
        "logs": f"{prefix}.pipeline_execution_logs",
    }

In [0]:
def create_tables(tables):
    """Create the schema and all tables if they do not exist yet (safe to run every time)."""
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {tables['schema']}")

    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {tables['bronze']} (
            `indicator`       STRUCT<id: STRING, value: STRING> COMMENT 'Indicator code and name as sent by the API',
            `country`         STRUCT<id: STRING, value: STRING> COMMENT 'Country/region code and name as sent by the API',
            `countryiso3code` STRING COMMENT 'ISO3 code (empty for some aggregates)',
            `date`            STRING COMMENT 'Year, raw string',
            `value`           STRING COMMENT 'Indicator value, raw text (cast in Silver)',
            `unit`            STRING,
            `obs_status`      STRING,
            `decimal`         STRING COMMENT 'Decimal precision, raw text',
            `_rescued_data`   STRING COMMENT 'Fields that did not match the Bronze schema (schema drift)',
            `record_hash`     STRING NOT NULL COMMENT 'SHA-256 of the raw record content',
            `source_file`     STRING NOT NULL COMMENT 'Raw file the record came from',
            `batch_id`        STRING NOT NULL COMMENT 'Batch / load identifier (pipeline parameter)',
            `load_type`       STRING COMMENT 'full or incremental',
            `load_timestamp`  TIMESTAMP NOT NULL COMMENT 'When the record was ingested into Bronze'
        ) USING DELTA
        COMMENT 'Bronze: raw World Bank indicator records. Key: (batch_id, record_hash)'
    """)

    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {tables['silver']} (
            `country_key`           STRING NOT NULL COMMENT 'Country id, or NAME_<name> when the API sends an empty id',
            `country_code`          STRING COMMENT 'World Bank country/region id',
            `country_iso3`          STRING COMMENT 'ISO3 code',
            `country_name`          STRING,
            `indicator_code`        STRING NOT NULL,
            `indicator_name`        STRING,
            `year`                  INT NOT NULL,
            `value`                 DOUBLE,
            `is_reported`           BOOLEAN COMMENT 'false when the API has no value for this country-year',
            `decimal_places`        INT,
            `source_batch_id`       STRING COMMENT 'Batch that last changed this row',
            `source_file`           STRING,
            `source_load_timestamp` TIMESTAMP COMMENT 'Bronze load time of the source record',
            `load_timestamp`        TIMESTAMP NOT NULL COMMENT 'When this row was last inserted/updated in Silver'
        ) USING DELTA
        COMMENT 'Silver: cleansed indicator values. Primary key: (country_key, indicator_code, year)'
    """)

    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {tables['revisions']} (
            `country_key`     STRING NOT NULL,
            `indicator_code`  STRING NOT NULL,
            `year`            INT NOT NULL,
            `old_value`       DOUBLE,
            `new_value`       DOUBLE,
            `change_type`     STRING COMMENT 'newly_reported, value_revised or value_removed',
            `old_batch_id`    STRING,
            `new_batch_id`    STRING NOT NULL,
            `load_timestamp`  TIMESTAMP NOT NULL
        ) USING DELTA
        COMMENT 'History of value changes applied to Silver by incremental loads'
    """)

    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {tables['quarantine']} (
            `layer`           STRING NOT NULL COMMENT 'raw_to_bronze or bronze_to_silver',
            `batch_id`        STRING,
            `source_file`     STRING,
            `reason`          STRING NOT NULL,
            `record_hash`     STRING,
            `raw_record`      STRING COMMENT 'The rejected record as JSON',
            `load_timestamp`  TIMESTAMP NOT NULL
        ) USING DELTA
        COMMENT 'Records rejected by validation (non-conforming / schema drift)'
    """)

    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {tables['logs']} (
            `log_id`            STRING NOT NULL,
            `run_id`            STRING NOT NULL COMMENT 'Same for all steps of one pipeline run',
            `layer`             STRING NOT NULL COMMENT 'raw_to_bronze or bronze_to_silver',
            `load_type`         STRING,
            `batch_id`          STRING,
            `source`            STRING COMMENT 'File or table/parameter processed',
            `start_time`        TIMESTAMP,
            `end_time`          TIMESTAMP,
            `duration_seconds`  DOUBLE,
            `status`            STRING COMMENT 'SUCCESS or FAILED',
            `rows_read`         BIGINT,
            `rows_inserted`     BIGINT,
            `rows_updated`      BIGINT,
            `rows_quarantined`  BIGINT,
            `message`           STRING,
            `load_timestamp`    TIMESTAMP NOT NULL
        ) USING DELTA
        COMMENT 'One row per file/table processed by every pipeline run'
    """)

    # Informational primary keys (supported on Unity Catalog only)
    for table, name, cols in [
        (tables["silver"], "pk_silver_indicator_values", "country_key, indicator_code, year"),
        (tables["logs"], "pk_pipeline_execution_logs", "log_id"),
    ]:
        try:
            spark.sql(f"ALTER TABLE {table} ADD CONSTRAINT {name} PRIMARY KEY ({cols})")
        except Exception:
            pass  # already exists, or not a Unity Catalog table


def drop_all_tables(tables):
    """Remove all pipeline tables. Only for a clean demo/test run."""
    for key in ["bronze", "silver", "revisions", "quarantine", "logs"]:
        spark.sql(f"DROP TABLE IF EXISTS {tables[key]}")

In [0]:
LOG_SCHEMA = StructType([
    StructField("log_id", StringType(), False),
    StructField("run_id", StringType(), False),
    StructField("layer", StringType(), False),
    StructField("load_type", StringType(), True),
    StructField("batch_id", StringType(), True),
    StructField("source", StringType(), True),
    StructField("start_time", TimestampType(), True),
    StructField("end_time", TimestampType(), True),
    StructField("duration_seconds", DoubleType(), True),
    StructField("status", StringType(), True),
    StructField("rows_read", LongType(), True),
    StructField("rows_inserted", LongType(), True),
    StructField("rows_updated", LongType(), True),
    StructField("rows_quarantined", LongType(), True),
    StructField("message", StringType(), True),
    StructField("load_timestamp", TimestampType(), False),
])


def write_log(tables, entry):
    row = tuple(entry.get(f.name) for f in LOG_SCHEMA.fields)
    spark.createDataFrame([row], LOG_SCHEMA).write.format("delta").mode("append").saveAsTable(tables["logs"])


def run_step(tables, run_id, layer, load_type, batch_id, source, step_fn):
    """Run one pipeline step and always write a log entry, whether it succeeds or fails.

    A failure is recorded and returned, it does not stop the other files of the batch.
    """
    entry = {
        "log_id": str(uuid.uuid4()),
        "run_id": run_id,
        "layer": layer,
        "load_type": load_type,
        "batch_id": batch_id,
        "source": source,
        "start_time": utc_now(),
    }
    try:
        metrics = step_fn() or {}
        entry.update(metrics)
        entry["status"] = "SUCCESS"
    except Exception as e:
        entry["status"] = "FAILED"
        entry["message"] = f"{type(e).__name__}: {e}"[:4000]
    finally:
        end = utc_now()
        entry["end_time"] = end
        entry["duration_seconds"] = round((end - entry["start_time"]).total_seconds(), 3)
        entry["load_timestamp"] = end
        write_log(tables, entry)
    print(f"[{entry['status']}] {layer} | {source} | read={entry.get('rows_read')} "
          f"inserted={entry.get('rows_inserted')} updated={entry.get('rows_updated')} "
          f"quarantined={entry.get('rows_quarantined')} | {entry.get('message') or ''}")
    return entry

In [0]:
def merge_into(merge_sql, target_table):
    """Run a MERGE statement and return (rows_inserted, rows_updated)."""
    result = spark.sql(merge_sql).collect()
    if result and "num_inserted_rows" in result[0].asDict():
        row = result[0].asDict()
        return int(row.get("num_inserted_rows") or 0), int(row.get("num_updated_rows") or 0)
    # Fallback: read the metrics of the last commit from the Delta history
    metrics = spark.sql(f"DESCRIBE HISTORY {target_table} LIMIT 1").collect()[0]["operationMetrics"]
    return int(metrics.get("numTargetRowsInserted", 0)), int(metrics.get("numTargetRowsUpdated", 0))


def quarantine_records(tables, df, layer, reason_col, raw_cols, load_ts):
    """Write rejected records to the quarantine table (MERGE, so a re-run adds no duplicates)."""
    q = (
        df.select(
            F.lit(layer).alias("layer"),
            F.col("batch_id"),
            F.col("source_file"),
            F.col(reason_col).alias("reason"),
            F.col("record_hash"),
            F.to_json(F.struct(*raw_cols)).alias("raw_record"),
            F.lit(load_ts).cast(TimestampType()).alias("load_timestamp"),
        )
        .dropDuplicates(["layer", "batch_id", "record_hash", "reason"])
    )
    q.createOrReplaceTempView("_quarantine_src")
    inserted, _ = merge_into(f"""
        MERGE INTO {tables['quarantine']} t
        USING _quarantine_src s
        ON  t.layer = s.layer AND t.batch_id <=> s.batch_id
        AND t.record_hash <=> s.record_hash AND t.reason = s.reason
        WHEN NOT MATCHED THEN INSERT *
    """, tables["quarantine"])
    return inserted

In [0]:
def load_file_to_bronze(tables, path, batch_id, load_type):
    """Raw file -> Bronze for one file."""
    load_ts = utc_now()
    file_name = os.path.basename(path)

    staged = (
        add_bronze_metadata(read_raw_file(path), file_name, batch_id, load_type, load_ts)
        .withColumn("_reject_reason", bronze_reject_reason())
    )

    stats = staged.agg(
        F.count(F.lit(1)).alias("rows_read"),
        F.count("_reject_reason").alias("rejected"),
        F.count("_rescued_data").alias("drift_rows"),
    ).collect()[0]

    if stats["rows_read"] == 0:
        raise ValueError(f"No records could be read from {path}")

    if stats["rejected"] > 0:
        quarantine_records(
            tables, staged.filter(F.col("_reject_reason").isNotNull()),
            "raw_to_bronze", "_reject_reason", RAW_FIELDS + ["_rescued_data"], load_ts
        )

    good = (
        staged.filter(F.col("_reject_reason").isNull())
        .drop("_reject_reason")
        .dropDuplicates(["batch_id", "record_hash"])
    )
    good.createOrReplaceTempView("_bronze_src")

    # Insert-only MERGE: the same raw record in the same batch is never loaded twice.
    # WITH SCHEMA EVOLUTION lets the table pick up new columns if bronze_schema is extended.
    inserted, _ = merge_into(f"""
        MERGE WITH SCHEMA EVOLUTION INTO {tables['bronze']} t
        USING _bronze_src s
        ON t.batch_id = s.batch_id AND t.record_hash = s.record_hash
        WHEN NOT MATCHED THEN INSERT *
    """, tables["bronze"])

    notes = []
    if stats["drift_rows"] > 0:
        notes.append(f"schema drift: {stats['drift_rows']} rows had unexpected/mismatched fields (kept in _rescued_data)")
    if stats["rejected"] > 0:
        notes.append(f"{stats['rejected']} rows quarantined")

    return {
        "rows_read": stats["rows_read"],
        "rows_inserted": inserted,
        "rows_updated": 0,
        "rows_quarantined": stats["rejected"],
        "message": "; ".join(notes) or None,
    }

In [0]:
def load_bronze_to_silver(tables, batch_id):
    """Bronze -> Silver for one batch (or batch_id='ALL' to rebuild from every batch)."""
    load_ts = utc_now()

    bronze = spark.table(tables["bronze"])
    if batch_id.upper() != "ALL":
        bronze = bronze.filter(F.col("batch_id") == batch_id)

    staged = to_silver(bronze, load_ts).withColumn("_reject_reason", silver_reject_reason())

    stats = staged.agg(
        F.count(F.lit(1)).alias("rows_read"),
        F.count("_reject_reason").alias("rejected"),
    ).collect()[0]

    if stats["rows_read"] == 0:
        raise ValueError(f"No Bronze records found for batch_id = {batch_id}")

    if stats["rejected"] > 0:
        quarantine_records(
            tables,
            staged.filter(F.col("_reject_reason").isNotNull()).withColumnRenamed("source_batch_id", "batch_id"),
            "bronze_to_silver", "_reject_reason",
            ["country_code", "country_name", "indicator_code", "year_raw", "value_raw", "decimal_raw"], load_ts
        )

    source = latest_per_key(staged.filter(F.col("_reject_reason").isNull())).select(*SILVER_COLUMNS)
    source.createOrReplaceTempView("_silver_src")

    # A row is only changed when the incoming record is at least as new as the one
    # already in Silver. This way re-running an old batch (backfill) cannot overwrite
    # newer values, and re-running the same batch changes nothing.
    change_condition = """
        s.source_load_timestamp >= t.source_load_timestamp AND (
            NOT (t.value <=> s.value)
            OR NOT (t.country_name <=> s.country_name)
            OR NOT (t.country_iso3 <=> s.country_iso3)
            OR NOT (t.indicator_name <=> s.indicator_name)
            OR NOT (t.decimal_places <=> s.decimal_places)
        )
    """

    # 1) Record value revisions before Silver is updated
    revisions, _ = merge_into(f"""
        MERGE INTO {tables['revisions']} r
        USING (
            SELECT s.country_key, s.indicator_code, s.year,
                   t.value AS old_value, s.value AS new_value,
                   CASE WHEN t.value IS NULL THEN 'newly_reported'
                        WHEN s.value IS NULL THEN 'value_removed'
                        ELSE 'value_revised' END AS change_type,
                   t.source_batch_id AS old_batch_id,
                   s.source_batch_id AS new_batch_id,
                   s.load_timestamp
            FROM _silver_src s
            JOIN {tables['silver']} t
              ON t.country_key = s.country_key AND t.indicator_code = s.indicator_code AND t.year = s.year
            WHERE s.source_load_timestamp >= t.source_load_timestamp
              AND NOT (t.value <=> s.value)
        ) c
        ON  r.country_key = c.country_key AND r.indicator_code = c.indicator_code
        AND r.year = c.year AND r.new_batch_id = c.new_batch_id
        WHEN NOT MATCHED THEN INSERT *
    """, tables["revisions"])

    # 2) Upsert into Silver
    inserted, updated = merge_into(f"""
        MERGE INTO {tables['silver']} t
        USING _silver_src s
        ON t.country_key = s.country_key AND t.indicator_code = s.indicator_code AND t.year = s.year
        WHEN MATCHED AND {change_condition} THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """, tables["silver"])

    notes = [f"{revisions} value revisions recorded"]
    if stats["rejected"] > 0:
        notes.append(f"{stats['rejected']} rows quarantined")

    return {
        "rows_read": stats["rows_read"],
        "rows_inserted": inserted,
        "rows_updated": updated,
        "rows_quarantined": stats["rejected"],
        "message": "; ".join(notes),
    }

In [0]:
def resolve_source_files(source_path, load_type, data_dir=None):
    """Turn the source_path parameter into a list of files.

    source_path can be empty (default files for the load type), a file, a folder
    (all .json files inside), a wildcard pattern, or several of these separated by commas.
    Relative paths are resolved against data_dir (default: the repo folder).
    """
    data_dir = data_dir or os.getcwd()
    entries = [s.strip() for s in (source_path or "").split(",") if s.strip()] or [DEFAULT_SOURCES[load_type]]
    files = []
    for entry in entries:
        if entry.startswith("dbfs:") or "://" in entry:
            files.append(entry)  # cloud/DBFS path, passed to Spark as it is
            continue
        path = entry if os.path.isabs(entry) else os.path.join(data_dir, entry)
        if os.path.isdir(path):
            files.extend(sorted(glob.glob(os.path.join(path, "*.json"))))
        else:
            files.extend(sorted(glob.glob(path)))
    return list(dict.fromkeys(files))


def default_batch_id(load_type, run_ts):
    # full load is one baseline; incremental pulls get one batch per day
    return "full_load" if load_type == "full" else f"incremental_{run_ts:%Y%m%d}"


def run_pipeline(load_type="incremental", source_path="", batch_id="", layer="all",
                 catalog="", schema=DEFAULT_SCHEMA, data_dir=None):
    """Run the pipeline with parameters.

    load_type   : 'full' or 'incremental'
    source_path : raw file(s) / folder to load. Empty = default files for the load type
    batch_id    : batch identifier. Empty = 'full_load' or 'incremental_<YYYYMMDD>'.
                  With layer='silver' it selects which Bronze batch to process
                  ('ALL' = every batch)
    layer       : 'all' (raw->bronze->silver), 'bronze' or 'silver'
    catalog / schema : where the tables live
    """
    load_type = (load_type or "").strip().lower()
    layer = (layer or "all").strip().lower()
    if load_type not in DEFAULT_SOURCES:
        raise ValueError("load_type must be 'full' or 'incremental'")
    if layer not in ("all", "bronze", "silver"):
        raise ValueError("layer must be 'all', 'bronze' or 'silver'")

    tables = get_tables(catalog, schema)
    create_tables(tables)

    run_id = str(uuid.uuid4())
    batch_id = (batch_id or "").strip() or default_batch_id(load_type, utc_now())
    if layer != "silver" and batch_id.upper() == "ALL":
        raise ValueError("batch_id 'ALL' can only be used with layer = 'silver'")

    print(f"run_id={run_id} | load_type={load_type} | batch_id={batch_id} | layer={layer} | tables={tables['schema']}")
    results = []

    if layer in ("all", "bronze"):
        files = resolve_source_files(source_path, load_type, data_dir)
        if not files:
            def no_files():
                raise FileNotFoundError(f"No source files found for source_path='{source_path or DEFAULT_SOURCES[load_type]}'")
            results.append(run_step(tables, run_id, "raw_to_bronze", load_type, batch_id,
                                    source_path or DEFAULT_SOURCES[load_type], no_files))
        for path in files:
            results.append(run_step(
                tables, run_id, "raw_to_bronze", load_type, batch_id, path,
                lambda p=path: load_file_to_bronze(tables, p, batch_id, load_type)
            ))

    if layer in ("all", "silver"):
        loaded = any(r["status"] == "SUCCESS" for r in results) or layer == "silver"
        if loaded:
            results.append(run_step(
                tables, run_id, "bronze_to_silver", load_type, batch_id,
                f"{tables['bronze']} [batch_id={batch_id}]",
                lambda: load_bronze_to_silver(tables, batch_id)
            ))

    failed = [r for r in results if r["status"] != "SUCCESS"]
    if failed:
        raise RuntimeError(
            f"{len(failed)} of {len(results)} step(s) failed in run {run_id}. "
            f"See {tables['logs']} for details."
        )
    return run_id, tables